In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

print("Import done ✔️")




Import done ✔️


# model a = visit risk classification 
#business_purpose = predict wether a hosporal visit represenrs Lowm Medium or Hogh operational and clinical risk.
 
 #Target : risk_score -  Low/Medium/High
 #split: Time-Based  -- earliest 80% train, latest 20% test
 
 #why time based split? Prevents data leakeage from future visits influencing predictions on past visits.

In [8]:
df = pd.read_csv("../outputs/model_table.csv")
df.head()

,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_LOS_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.120000,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.750000,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.460000,0.242556,1,3,1
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,1,0,0,60,3,30.056667,0.256876,1,3,0
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,1,0,0,330,9,29.030000,0.157496,1,3,1


In [9]:
#load the dataset
df = pd.read_csv("../outputs/model_table.csv", parse_dates=["registration_date", "visit_date", "billing_date"])
print("Shape:", df.shape)
df.head()

Shape: (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_LOS_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.120000,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.750000,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.460000,0.242556,1,3,1
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,1,0,0,60,3,30.056667,0.256876,1,3,0
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,1,0,0,330,9,29.030000,0.157496,1,3,1


In [11]:
risk_target = "risk_score"
risk_features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "length_of_stay_hours",
    "days_since_registration",
    "visit_frequency",
    "avg_LOS_per_patient",
    "visit_month",
    "visit_dayofweek"
]
risk_df = df[risk_features + [risk_target , "visit_date"]].copy()
risk_df = risk_df.dropna(subset = [risk_target, "visit_date"])
print("Risk_dataset shape: ", risk_df.shape)
risk_df.head()

Risk_dataset shape:  (25000, 16)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,days_since_registration,visit_frequency,avg_LOS_per_patient,visit_month,visit_dayofweek,risk_score,visit_date
0,15,F,Mumbai,CareOne,0,General,OPD,105,9.63,5,4,21.120000,1,3,Low,2026-01-01
1,3,M,Bangalore,CareOne,0,ICU,ICU,112,59.60,141,8,23.750000,1,3,High,2026-01-01
2,44,M,Pune,MediCareX,1,ICU,ER,150,59.28,165,3,32.460000,1,3,High,2026-01-01
3,47,F,Delhi,CareOne,1,General,OPD,145,25.15,60,3,30.056667,1,3,Medium,2026-01-01
4,14,F,Chennai,SecureLife,1,Cardiology,ER,148,42.88,330,9,29.030000,1,3,Medium,2026-01-01


In [14]:
risk_df = risk_df.sort_values("visit_date").reset_index(drop = True)
split_idx = int(len(risk_df)*0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print(f"train shape: {X_train_risk.shape}")
print(f"test shape: {X_test_risk.shape}")
print("Train period: ",risk_train["visit_date"].min().date(),"->", risk_train["visit_date"].max().date())
print("Test Period: ", risk_test["visit_date"].min().date(),"->", risk_test["visit_date"].max().date())

train shape: (20000, 14)
test shape: (5000, 14)
Train period:  2025-01-21 -> 2026-01-02
Test Period:  2026-01-02 -> 2026-01-20


In [ ]:
#processing pipeline

# two transformers 
# Numrical - SimpleImputer(median Strategy)
# Categorical - SimpleImputer + OneHotEncoder

#combined with coloumn transfer. Entire pipeline get saved with the model--no separate preprocessing step at inference time.

In [18]:
risk_numeric_features = [
    "age",
    "chronic_flag",
    "length_of_stay_hours",
    "days_since_registration",
    "doctor_id",
    "avgerage_LOS_per_patient",
    "visit_month",
    "visit_dayofweek"
]

risk_categorical_features  = [
    "gender",
    "city",
    "insurance_provider",
    "department",
    "visit_type"
    
]

risk_numeric_transformer = Pipeline(steps=[("imputer"), SimpleImputer(strategy="median")])
risk_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("OneHot", OneHotEncoder(handle_unknown="ignore") )
    ]
)


risk_processor = ColumnTransformer(transformers=[
    ("num",risk_numeric_transformer, risk_numeric_features),
    ("cat", risk_categorical_transformer, risk_categorical_features)
])

print("Processing Pipeline ready! ✔️")


Processing Pipeline ready! ✔️
